# 8.6 SVM – Exercises

## Exercise 1: Random dataset following a quadratic distribution, classified with SVM

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

rng = np.random.default_rng(0)
X1 = rng.uniform(-3, 3, 300)
X2 = X1**2 + rng.normal(0, 0.8, 300)            # points scattered around y = x^2
X = np.c_[X1, X2 + rng.choice([-2.5, 2.5], 300)]  # shift half above / half below the parabola
y = (X[:, 1] > X[:, 0]**2).astype(int)             # class = above or below the parabola

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
for k in ['linear', 'poly', 'rbf']:
    m = SVC(kernel=k, degree=2).fit(X_train, y_train)
    print(k, 'accuracy:', accuracy_score(y_test, m.predict(X_test)))

In [ ]:
model = SVC(kernel='rbf').fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))
xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 300), np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.3); plt.scatter(X[:,0], X[:,1], c=y, edgecolor='k', s=20)
plt.title('SVM (rbf) on quadratic data'); plt.show()

## Exercise 2: Spam / not-spam email classification with SVM
No dataset is given, so features (word frequency of spam words, message length) are simulated.

In [ ]:
import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

rng = np.random.default_rng(1)
n = 500
spam = pd.DataFrame({'word_freq': rng.normal(6, 2, n), 'msg_length': rng.normal(300, 80, n), 'label': 1})
ham  = pd.DataFrame({'word_freq': rng.normal(1.5, 1, n), 'msg_length': rng.normal(600, 200, n), 'label': 0})
emails = pd.concat([spam, ham]).sample(frac=1, random_state=1).reset_index(drop=True)
emails.head()

In [ ]:
x = emails[['word_freq', 'msg_length']]; y = emails['label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=5)
sc = StandardScaler(); x_train = sc.fit_transform(x_train); x_test = sc.transform(x_test)

model = SVC(kernel='rbf', random_state=0).fit(x_train, y_train)
pred = model.predict(x_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['not spam', 'spam']))